In [ ]:
# Find the pipeline package without walking the DICOM tree.
import sys, os, time, gc, json
from pathlib import Path

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

def walk(root="/kaggle/input", max_depth=8, suffixes=None):
    base = len(Path(root).parts)
    for dirpath, dirnames, filenames in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - base >= max_depth:
            dirnames[:] = []
        else:
            dirnames[:] = [d for d in dirnames if not d.lower().endswith(("_series", "_images"))]
        for f in filenames:
            if suffixes is None or Path(f).suffix.lower() in suffixes:
                yield here / f

PKG = next((p.parent.parent for p in walk(suffixes=(".py",))
            if p.name == "targets.py" and p.parent.name == "rsna_knee"), None)
if PKG is None:
    raise FileNotFoundError("attach the rsna-knee-pipeline dataset (v2, it must contain rsna_knee/targets.py)")
sys.path.insert(0, str(PKG))

import numpy as np
import pandas as pd
from rsna_knee.report_labels import TARGETS
log(f"pipeline: {PKG}")


In [ ]:
# ---- settings for this run (defaults: the reported family r50v3) ---------------
ENCODER = "rad_resnet50"          # or a timm name, e.g. "convnext_tiny.fb_in22k_ft_in1k"
TAG = "r50v3"                     # family name; checkpoints are v2_<TAG>_fold<k>.pt
SEED = 2026
FOLDS = [0]                       # one fold per notebook run
FULL_DATA = False                 # True: one extra model on all studies (v2_<TAG>_full.pt)
EPOCHS = 8
CACHE = "v3"                      # "v1": old 288 px cache   "v3": cache from notebook 06
GROUP = 3                         # 3: 2.5D windows   1: single slices (8 per slot)
SCHEME = "flat"                   # teacher share: "flat" (0.5), "grouped", "labels_only"
EXTRA_LABELS = []                 # other public report-label csv files, e.g. ["llm_labels_v4_blend.csv"]
SLOT_PRIOR = True                 # radiological finding-to-slot prior
CROP = True                       # random crop in training (v3: 224 of 256 px); False: whole slice
LOCAL_COPY = True                 # copy the v3 cache to the local disk before training (much faster reads)
EXTRA_TEACHERS = []               # round 2: e.g. ["oof_v2_radr50_f0.csv", "oof_v2_radr50_f1234.csv"]
LR_ENCODER = 2e-4 if ENCODER == "rad_resnet50" else 1e-4
# ---------------------------------------------------------------------------

from rsna_knee.data import CacheConfig, load_cache
from rsna_knee.targets import build_targets_v2, gold_auc_table, load_default_sources, TEACHER_SHARE
from rsna_knee.train2 import TrainConfigV2, train_one, write_oof, auc_table

WEIGHTS = None
if ENCODER == "rad_resnet50":
    WEIGHTS = next((str(p) for p in walk(suffixes=(".pt", ".pth")) if p.name == "ResNet50.pt"), None)
    if WEIGHTS is None:
        raise FileNotFoundError("attach Kaggle dataset marwanmath/resnet-50-radimagenet-marwan (ResNet50.pt)")
    log(f"RadImageNet weights: {WEIGHTS}")
else:
    # timm downloads the weights: turn Internet ON for this training notebook,
    # or attach a .pth whose file name starts with the encoder name.
    WEIGHTS = next((str(p) for p in walk(suffixes=(".pth", ".pt", ".safetensors"))
                    if p.name.startswith(ENCODER.split(".")[0])), None)
    log(f"timm weights file: {WEIGHTS} (None means download)")

FOLDS_CSV = next(p for p in walk(suffixes=(".csv",)) if p.name == "train_folds.csv")
train = pd.read_csv(FOLDS_CSV)
if CACHE == "v1":
    CFG = CacheConfig(px=288, slices_per_slot=12, group=3, crop_mm=130.0)
    CACHE_DIRS = sorted({p.parent for p in walk(suffixes=(".npy",)) if p.name.startswith("train_")})
    array, mask, studies = load_cache(CACHE_DIRS, "train", CFG)
    assert studies == train.StudyInstanceUID.tolist(), "cache order does not match train_folds.csv"
    IMG = 256 if CROP else 288
else:
    from rsna_knee.cache3 import load_cache3, copy_local, read_speed
    CACHE_DIRS = sorted({p.parent for p in walk(suffixes=(".npy",)) if p.name.startswith("c3_train_")})
    if LOCAL_COPY:
        CACHE_DIRS = copy_local(CACHE_DIRS, log=log)
    array, mask, missing = load_cache3(CACHE_DIRS, "train", train.StudyInstanceUID.tolist())
    read_speed(array, log=log)
    if missing:
        raise FileNotFoundError(f"{len(missing)} studies missing from the v3 cache: attach every shard")
    IMG = 224 if CROP else 256
EMPTY = mask.sum(1) == 0
log(f"cache {array.shape}, slots filled {mask.mean():.3f}, empty studies {EMPTY.sum()}")


In [ ]:
labels, teachers, lab_path, oof_path = load_default_sources(EXTRA_TEACHERS, EXTRA_LABELS)
log(f"report labels: {lab_path}\nteachers: {[str(oof_path)] + EXTRA_TEACHERS}")

# Quality of the training targets themselves on gold-58, before gold overrides them.
t_nogold, _, _, _ = build_targets_v2(train, labels, teachers, scheme=SCHEME, use_gold=False)
lab_only, _, _, _ = build_targets_v2(train, labels, [], scheme="labels_only", use_gold=False)
cmp = gold_auc_table(train, lab_only).rename(columns={"auc": "labels"})
cmp["targets"] = gold_auc_table(train, t_nogold)["auc"]
log("\n" + cmp.round(3).to_string(index=False))
log(f"gold macro  labels {cmp.labels.mean():.4f}  ->  targets {cmp.targets.mean():.4f}")

targets, weights, explicit, is_gold = build_targets_v2(train, labels, teachers, scheme=SCHEME)
gold = train[list(TARGETS)].to_numpy(dtype=np.float32)
gold[~is_gold] = np.nan


In [ ]:
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
tcfg = TrainConfigV2(encoder=ENCODER, encoder_weights=WEIGHTS, epochs=EPOCHS,
                     lr_encoder=LR_ENCODER, target_scheme=SCHEME, tag=TAG,
                     cache=CACHE, group=GROUP, img=IMG,
                     train_windows=4 if GROUP == 3 else 8, eval_windows=16 if GROUP == 3 else 8,
                     slot_prior=SLOT_PRIOR, seed=SEED)
deadline = T0 + tcfg.time_budget_s
oof = np.full((len(train), len(TARGETS)), np.nan, dtype=np.float32)
hist = {}
for fold in FOLDS:
    if time.time() > deadline:
        log(f"no time left for fold {fold}"); break
    tr = np.flatnonzero((train.fold.values != fold) & ~EMPTY)
    va = np.flatnonzero(train.fold.values == fold)
    log(f"==== fold {fold}: {len(tr)} train / {len(va)} valid")
    p, h = train_one(tcfg, array, mask, tr, va, targets, weights, explicit, gold,
                     OUT / f"v2_{TAG}_fold{fold}.pt", deadline=deadline, log=log)
    if p is not None:
        oof[va] = p
    hist[f"fold{fold}"] = h
    gc.collect()

if FULL_DATA and time.time() < deadline:
    log("==== full-data model")
    _, h = train_one(tcfg, array, mask, np.flatnonzero(~EMPTY), [], targets, weights, None, None,
                     OUT / f"v2_{TAG}_full.pt", deadline=deadline, log=log)
    hist["full"] = h

RUN = f"{TAG}_f{''.join(map(str, FOLDS))}"
json.dump(hist, open(OUT / f"history_{RUN}.json", "w"), indent=1)

# provenance: settings, package versions and a hash of every checkpoint written by this run
import hashlib, platform, torch, timm
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
manifest = {"config": {k: v for k, v in vars(tcfg).items() if k != "encoder_weights"},
            "folds": FOLDS, "scheme": SCHEME, "crop": CROP, "cache": CACHE,
            "labels": str(lab_path), "teachers": [str(oof_path)] + EXTRA_TEACHERS,
            "inputs_sha256": {str(q): sha256(q) for q in [lab_path, oof_path] + ([WEIGHTS] if WEIGHTS else [])},
            "versions": {"python": platform.python_version(), "torch": torch.__version__,
                         "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__},
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "checkpoints": {p.name: sha256(p) for p in sorted(OUT.glob(f"v2_{TAG}_*.pt"))}}
json.dump(manifest, open(OUT / f"run_{RUN}.json", "w"), indent=1, default=str)
log(f"manifest: {manifest['checkpoints']}")


In [ ]:
done = np.isfinite(oof).all(axis=1)
if done.any():
    write_oof(train, oof, OUT / f"oof_v2_{TAG}_f{''.join(map(str, FOLDS))}.csv")
    ex = auc_table(explicit[done], oof[done])
    log(f"OOF vs explicit report statements ({done.sum()} studies): macro {ex['macro']:.4f}")
    g = gold_auc_table(train[done].reset_index(drop=True), oof[done])
    log("\n" + g.round(3).to_string(index=False))
    log(f"OOF gold macro: {g.auc.mean():.4f}   (labels alone were {cmp.labels.mean():.4f})")
log("done. Save this version's output as a Dataset, e.g. rsna-knee-v2-" + TAG)
